# Milestone 5 — Hybrid Retrieval

Notebook này **chỉ load** BM25/FAISS artifacts từ M3–M4, chuẩn hóa hai retriever và hợp nhất Top-N bằng Reciprocal Rank Fusion (RRF). Không rebuild index, không tích hợp LLM.

## 1. Setup Colab và Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa')
assert PROJECT_ROOT.exists(), f'Không tìm thấy repo: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)

Project root: /content/drive/MyDrive/Master/IT2043/Project/Final_Project/vn-landlaw-qa


In [11]:
# Cài dependency runtime; không build lại BM25 hoặc FAISS.
%pip install -q -r requirements/retrieval.txt

## 2. Kiểm tra và load artifacts M3–M4

In [12]:
import json

CHUNKS_PATH = PROJECT_ROOT / 'data/processed/chunks.json'
BM25_INDEX_PATH = PROJECT_ROOT / 'indexes/bm25/bm25_index.json.gz'
FAISS_DIR = PROJECT_ROOT / 'indexes/faiss'
REQUIRED = [
    CHUNKS_PATH, BM25_INDEX_PATH, FAISS_DIR / 'faiss.index',
    FAISS_DIR / 'vector_metadata.json', FAISS_DIR / 'embedding_manifest.json',
]
missing = [str(path) for path in REQUIRED if not path.exists()]
assert not missing, 'Thiếu artifacts M3/M4:\n' + '\n'.join(missing)

chunks = json.loads(CHUNKS_PATH.read_text(encoding='utf-8'))
print(f'Chunks: {len(chunks):,}')
print('Artifacts hợp lệ; notebook sẽ load, không rebuild.')

Chunks: 921
Artifacts hợp lệ; notebook sẽ load, không rebuild.


In [13]:
from landlaw_rag.retrieval import (
    BM25Index, VectorIndex, BM25Retriever, VectorRetriever,
    HybridConfig, HybridRetriever, apply_version_boost, detect_version_intent,
)

bm25_index = BM25Index.load(BM25_INDEX_PATH)
vector_index = VectorIndex.load(
    FAISS_DIR, device='cuda', load_encoder=True, chunks_path=CHUNKS_PATH
)
print(f'BM25 documents: {len(bm25_index.documents):,}')
print(f'FAISS vectors: {vector_index.index.ntotal:,}; dimension: {vector_index.index.d}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BM25 documents: 921
FAISS vectors: 921; dimension: 768


## 3. Tạo Hybrid Retriever

Hai adapter có cùng interface `search(query, top_k)`. RRF dùng công thức `sum(1 / (rrf_k + rank))`; kết quả được deduplicate theo `chunk_id`. Sau RRF, `apply_version_boost()` ưu tiên Luật 2013/2024 bằng metadata. Query không nêu năm mặc định ưu tiên Luật 2024; query so sánh giữ evidence của cả hai phiên bản.

In [14]:
bm25_retriever = BM25Retriever(bm25_index)
vector_retriever = VectorRetriever(vector_index)
hybrid_retriever = HybridRetriever(
    bm25=bm25_retriever,
    vector=vector_retriever,
    config=HybridConfig(bm25_top_n=20, vector_top_n=20, final_top_k=8, rrf_k=60),
)

def hybrid_search(query: str, top_k: int = 8) -> list[dict]:
    """API M5 dùng lại được ở M6: trả context đã fusion và xếp hạng."""
    return hybrid_retriever.search(query, top_k=top_k)

In [15]:
# Smoke test schema và deduplication.
sample = hybrid_search('Khoản 2 Điều 57 Luật 45/2013/QH13', top_k=5)
required_fields = {'chunk_id', 'text', 'metadata', 'bm25_score', 'vector_score', 'rrf_score', 'version_score', 'final_score', 'rank'}
assert sample and all(required_fields <= item.keys() for item in sample)
assert len({item['chunk_id'] for item in sample}) == len(sample)
sample[0]

{'chunk_id': '45-2013-qh13-d0347d32d57475a9',
 'text': 'Luật Đất đai số 45/2013/QH13\nChương XIV - ĐIỀU KHOẢN THI HÀNH\nĐiều 210. Điều khoản chuyển tiếp\n2. Trường hợp nhà đầu tư được Nhà nước cho thuê đất thu tiền thuê đất hàng năm để đầu tư xây dựng, kinh doanh kết cấu hạ tầng khu công nghiệp, cụm công nghiệp khu chế xuất mà đã cho thuê lại đất có kết cấu hạ tầng theo hình thức trả tiền thuê đất một lần cho cả thời gian thuê trước ngày Luật này có hiệu lực thi hành thì phải nộp tiền cho Nhà nước theo quy định của Chính phủ; người thuê lại đất có quyền và nghĩa vụ như được Nhà nước cho thuê đất trả tiền một lần cho cả thời gian thuê sau khi chủ đầu tư đã nộp đủ tiền thuê đất vào ngân sách nhà nước.',
 'metadata': {'document_id': '45-2013-QH13',
  'document_title': 'Luật Đất đai số 45/2013/QH13',
  'chapter': 'Chương XIV - ĐIỀU KHOẢN THI HÀNH',
  'section': None,
  'article': 'Điều 210',
  'article_title': 'Điều khoản chuyển tiếp',
  'clause': 'Khoản 2',
  'point': None,
  'source_file

## 4. So sánh BM25 vs Vector vs Hybrid

In [16]:
import pandas as pd

def citation(metadata: dict) -> str:
    fields = ('document_title', 'article', 'clause', 'point')
    return ' – '.join(str(metadata.get(field)) for field in fields if metadata.get(field))

def compare_retrievers(query: str, top_k: int = 5) -> pd.DataFrame:
    rows = []
    result_sets = {
        'BM25': bm25_retriever.search(query, top_k),
        'Vector': vector_retriever.search(query, top_k),
        'Hybrid': hybrid_search(query, top_k),
    }
    for method, results in result_sets.items():
        for item in results:
            rows.append({
                'method': method,
                'rank': item['rank'],
                'chunk_id': item['chunk_id'],
                'citation': citation(item['metadata']),
                'bm25_score': item.get('bm25_score', item.get('score') if method == 'BM25' else None),
                'vector_score': item.get('vector_score', item.get('score') if method == 'Vector' else None),
                'rrf_score': item.get('rrf_score'),
                'version_score': item.get('version_score'),
                'final_score': item.get('final_score'),
            })
    return pd.DataFrame(rows)

TEST_QUERIES = [
    'Người dân được bồi thường như thế nào khi Nhà nước thu hồi đất ở?',
    'Điều 79 Luật Đất đai 2013 quy định gì?',
    'Điều 98 Luật Đất đai 2024 quy định gì?',
    'So sánh quy định bồi thường đất ở giữa 2013 và 2024.',
]
for query in TEST_QUERIES:
    print('\nQUERY:', query)
    display(compare_retrievers(query, top_k=5))


QUERY: Người dân được bồi thường như thế nào khi Nhà nước thu hồi đất ở?


,method,rank,chunk_id,citation,bm25_score,vector_score,rrf_score,version_score,final_score
0,BM25,1,45-2013-qh13-9460662138bec4b4,Luật Đất đai số 45/2013/QH13 – Điều 79,24.994972,NaN,NaN,NaN,NaN
1,BM25,2,31-2024-qh15-931090d983cba734,Luật Đất đai số 31/2024/QH15 – Điều 111 – Khoản 5,19.433692,NaN,NaN,NaN,NaN
2,BM25,3,31-2024-qh15-fc5e8fe5721cf76a,Luật Đất đai số 31/2024/QH15 – Điều 3 – Khoản 39,18.719071,NaN,NaN,NaN,NaN
3,BM25,4,31-2024-qh15-10c6ac8ce0f00cc4,Luật Đất đai số 31/2024/QH15 – Điều 102,18.014922,NaN,NaN,NaN,NaN
4,BM25,5,45-2013-qh13-59c4ceeaa441bd9d,Luật Đất đai số 45/2013/QH13 – Điều 81,17.959738,NaN,NaN,NaN,NaN
5,Vector,1,45-2013-qh13-b974bc7828208af6,Luật Đất đai số 45/2013/QH13 – Điều 74,NaN,0.883369,NaN,NaN,NaN
6,Vector,2,31-2024-qh15-ade923b040ef124a,Luật Đất đai số 31/2024/QH15 – Điều 97,NaN,0.882229,NaN,NaN,NaN
7,Vector,3,31-2024-qh15-b1d8fe553204a515,Luật Đất đai số 31/2024/QH15 – Điều 99,NaN,0.881893,NaN,NaN,NaN
8,Vector,4,45-2013-qh13-9460662138bec4b4,Luật Đất đai số 45/2013/QH13 – Điều 79,NaN,0.879732,NaN,NaN,NaN
9,Vector,5,31-2024-qh15-ead1962c213edc06,Luật Đất đai số 31/2024/QH15 – Điều 98,NaN,0.877632,NaN,NaN,NaN



QUERY: Điều 79 Luật Đất đai 2013 quy định gì?


,method,rank,chunk_id,citation,bm25_score,vector_score,rrf_score,version_score,final_score
0,BM25,1,45-2013-qh13-9460662138bec4b4,Luật Đất đai số 45/2013/QH13 – Điều 79,17.965629,NaN,NaN,NaN,NaN
1,BM25,2,31-2024-qh15-37125255e44c6d64,Luật Đất đai số 31/2024/QH15 – Điều 79 – Khoản 31,15.649950,NaN,NaN,NaN,NaN
2,BM25,3,31-2024-qh15-cd5c5ab64c3e8e0f,Luật Đất đai số 31/2024/QH15 – Điều 79 – Khoản 29,15.489386,NaN,NaN,NaN,NaN
3,BM25,4,31-2024-qh15-7a4bfd8d0c36bd80,Luật Đất đai số 31/2024/QH15 – Điều 79 – Khoản 24,15.458626,NaN,NaN,NaN,NaN
4,BM25,5,31-2024-qh15-5ac9c8a5d1356f3f,Luật Đất đai số 31/2024/QH15 – Điều 79 – Khoản 8,15.439505,NaN,NaN,NaN,NaN
5,Vector,1,45-2013-qh13-34b35a6d8317da14,Luật Đất đai số 45/2013/QH13 – Điều 170,NaN,0.867379,NaN,NaN,NaN
6,Vector,2,45-2013-qh13-7f9ba4cf7f870596,Luật Đất đai số 45/2013/QH13 – Điều 197,NaN,0.866864,NaN,NaN,NaN
7,Vector,3,45-2013-qh13-f4ec5ada52fc3b2f,Luật Đất đai số 45/2013/QH13 – Điều 96,NaN,0.866175,NaN,NaN,NaN
8,Vector,4,45-2013-qh13-d95141cd19f9c851,Luật Đất đai số 45/2013/QH13 – Điều 198,NaN,0.865923,NaN,NaN,NaN
9,Vector,5,45-2013-qh13-db897203c81b6a11,Luật Đất đai số 45/2013/QH13 – Điều 82,NaN,0.865157,NaN,NaN,NaN



QUERY: Điều 98 Luật Đất đai 2024 quy định gì?


,method,rank,chunk_id,citation,bm25_score,vector_score,rrf_score,version_score,final_score
0,BM25,1,31-2024-qh15-ead1962c213edc06,Luật Đất đai số 31/2024/QH15 – Điều 98,21.883358,NaN,NaN,NaN,NaN
1,BM25,2,45-2013-qh13-3125cc5a59faa920,Luật Đất đai số 45/2013/QH13 – Điều 98 – Khoản 1,18.003508,NaN,NaN,NaN,NaN
2,BM25,3,45-2013-qh13-a7dadc3f484a6480,Luật Đất đai số 45/2013/QH13 – Điều 98 – Khoản 2,17.719339,NaN,NaN,NaN,NaN
3,BM25,4,45-2013-qh13-6bb516fa3ab600c2,Luật Đất đai số 45/2013/QH13 – Điều 98 – Khoản 3,17.659557,NaN,NaN,NaN,NaN
4,BM25,5,45-2013-qh13-eece22662d6a73fb,Luật Đất đai số 45/2013/QH13 – Điều 98 – Khoản 4,16.896757,NaN,NaN,NaN,NaN
5,Vector,1,31-2024-qh15-3e9a18ef495621ec,Luật Đất đai số 31/2024/QH15 – Điều 8,NaN,0.869180,NaN,NaN,NaN
6,Vector,2,31-2024-qh15-183e25bc5d9b79c0,Luật Đất đai số 31/2024/QH15 – Điều 90 – Khoản 8,NaN,0.868392,NaN,NaN,NaN
7,Vector,3,31-2024-qh15-ead1962c213edc06,Luật Đất đai số 31/2024/QH15 – Điều 98,NaN,0.867717,NaN,NaN,NaN
8,Vector,4,43-2024-qh15-87bcc97045c7e381,"Luật số 43/2024/QH15 sửa đổi, bổ sung một số đ...",NaN,0.861488,NaN,NaN,NaN
9,Vector,5,31-2024-qh15-052991f070869f34,Luật Đất đai số 31/2024/QH15 – Điều 127 – Khoản 1,NaN,0.860337,NaN,NaN,NaN



QUERY: So sánh quy định bồi thường đất ở giữa 2013 và 2024.


,method,rank,chunk_id,citation,bm25_score,vector_score,rrf_score,version_score,final_score
0,BM25,1,31-2024-qh15-8f8039a18bf937b6,Luật Đất đai số 31/2024/QH15 – Điều 158 – Khoản 6,21.108326,NaN,NaN,NaN,NaN
1,BM25,2,31-2024-qh15-ede686a543429cbe,Luật Đất đai số 31/2024/QH15 – Điều 158 – Khoản 5,15.641152,NaN,NaN,NaN,NaN
2,BM25,3,254-2025-qh15-d455473a99ee4332,Nghị quyết số 254/2025/QH15 – Điều 3 – Khoản 7,14.320211,NaN,NaN,NaN,NaN
3,BM25,4,31-2024-qh15-1a0daeed56300854,Luật Đất đai số 31/2024/QH15 – Điều 111 – Khoản 8,14.070217,NaN,NaN,NaN,NaN
4,BM25,5,31-2024-qh15-e032843841748237,Luật Đất đai số 31/2024/QH15 – Điều 94 – Khoản 5,11.560848,NaN,NaN,NaN,NaN
5,Vector,1,45-2013-qh13-b974bc7828208af6,Luật Đất đai số 45/2013/QH13 – Điều 74,NaN,0.862428,NaN,NaN,NaN
6,Vector,2,45-2013-qh13-a9ee54d440db11b4,Luật Đất đai số 45/2013/QH13 – Điều 3 – Khoản 12,NaN,0.857726,NaN,NaN,NaN
7,Vector,3,31-2024-qh15-ead1962c213edc06,Luật Đất đai số 31/2024/QH15 – Điều 98,NaN,0.857128,NaN,NaN,NaN
8,Vector,4,45-2013-qh13-db897203c81b6a11,Luật Đất đai số 45/2013/QH13 – Điều 82,NaN,0.856626,NaN,NaN,NaN
9,Vector,5,45-2013-qh13-911130f17236567b,Luật Đất đai số 45/2013/QH13 – Điều 93,NaN,0.855951,NaN,NaN,NaN


## 5. Ghi chú cho Milestone 6

M6 chỉ cần gọi `hybrid_search(question, top_k)` và dùng `text` cùng `metadata` của từng kết quả để dựng context/citation. `bm25_rank`, `vector_rank`, `rrf_score`, `version_score` và `final_score` được giữ lại để truy vết. Có thể tái sử dụng độc lập `detect_version_intent()` và `apply_version_boost()`. Reranker có thể được chèn sau bước này mà không đổi interface retriever.